## Load Dataset

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)


In [2]:
dataset = load_dataset("banking77")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


### Check Label names

In [3]:
label_names = dataset['train'].features['label'].names
print(f"\nTotal classes: {len(label_names)}")
print(f"Sample classes: {label_names[:10]}")
# ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', ...]

print(f"\nTrain size: {len(dataset['train'])}")
print(f"Test size:  {len(dataset['test'])}")


Total classes: 77
Sample classes: ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire']

Train size: 10003
Test size:  3080


## Tokenize and model

In [14]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names)
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def preprocess_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=128
    )

tokenized_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

### Metrics

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

### Training args and Trainer

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    logging_dir='./logs',
    logging_steps=50,
    report_to='none', save_strategy="no"
)

 trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
) 

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Train and evaluate

In [18]:
trainer.train()


results = trainer.evaluate()
print(f"\nFinal Accuracy: {results['eval_accuracy']:.4f}")
print(f"Final F1 Score: {results['eval_f1']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,6.128375,5.582349,0.562338,0.504007
2,3.296122,2.962011,0.752273,0.721653
3,1.982560,1.869321,0.836364,0.823957
4,1.377997,1.446343,0.857792,0.849774
5,1.218207,1.325627,0.869156,0.863603


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


Final Accuracy: 0.8692
Final F1 Score: 0.8636


### Save

In [19]:
trainer.save_model("/kaggle/working/banking77-classifier")
tokenizer.save_pretrained("/kaggle/working/banking77-classifier")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/banking77-classifier/tokenizer_config.json',
 '/kaggle/working/banking77-classifier/tokenizer.json')

### Pipeline

In [20]:
classifier = pipeline(
    "text-classification",
    model="/kaggle/working/banking77-classifier",
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Predictor

In [21]:
def predictor(input_text, org_label=None):
    print(f"Input Ticket:\n{input_text}")
    print()
    result = classifier(input_text)

    pred_index     = int(result[0]['label'].split("_")[-1])
    pred_label     = label_names[pred_index]

    if org_label is not None:
        org_label_name = label_names[org_label]
        print(f"Original Label:  {org_label} , Original Label Decoded:  {org_label_name}")

    print(f"Predicted Label: {pred_index} , Predicted Label Decoded: {pred_label}")
    print(f"Confidence: {result[0]['score']:.2%}")
    print()

### Test on dataset samples

In [22]:
for i in [0, 5, 10, 20]:
    sample = dataset['test'][i]
    predictor(sample['text'], sample['label'])

Input Ticket:
How do I locate my card?

Original Label:  11 , Original Label Decoded:  card_arrival
Predicted Label: 11 , Predicted Label Decoded: card_arrival
Confidence: 29.99%

Input Ticket:
When will I get my card?

Original Label:  11 , Original Label Decoded:  card_arrival
Predicted Label: 12 , Predicted Label Decoded: card_delivery_estimate
Confidence: 67.73%

Input Ticket:
How do I track my card?

Original Label:  11 , Original Label Decoded:  card_arrival
Predicted Label: 11 , Predicted Label Decoded: card_arrival
Confidence: 72.89%

Input Ticket:
I did not get my card yet, is it lost?

Original Label:  11 , Original Label Decoded:  card_arrival
Predicted Label: 11 , Predicted Label Decoded: card_arrival
Confidence: 47.51%



### custome dataset test'

In [25]:

print("=" * 60)
print("Custom Support Email Predictions:")
print("=" * 60)
custom_emails = [
    "I have been trying to activate my card for 3 days now and it still does not work. Please help.",
    "Can you explain why I was charged twice for the same transaction on my account?",
    "I would like to know the daily limit for international transfers.",
    "My card was declined at the ATM even though I have sufficient balance.",
    "I need to update my phone number linked to my account."
]

for email in custom_emails:
    predictor(email)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Custom Support Email Predictions:
Input Ticket:
I have been trying to activate my card for 3 days now and it still does not work. Please help.

Predicted Label: 0 , Predicted Label Decoded: activate_my_card
Confidence: 86.24%

Input Ticket:
Can you explain why I was charged twice for the same transaction on my account?

Predicted Label: 63 , Predicted Label Decoded: transaction_charged_twice
Confidence: 90.40%

Input Ticket:
I would like to know the daily limit for international transfers.

Predicted Label: 64 , Predicted Label Decoded: transfer_fee_charged
Confidence: 14.03%

Input Ticket:
My card was declined at the ATM even though I have sufficient balance.

Predicted Label: 26 , Predicted Label Decoded: declined_cash_withdrawal
Confidence: 75.38%

Input Ticket:
I need to update my phone number linked to my account.

Predicted Label: 30 , Predicted Label Decoded: edit_personal_details
Confidence: 74.34%

